# 02 - Data Cleaning: House Price Prediction

Based on the EDA findings:
- Most missing values mean "feature does not exist" (Pool, Alley, Fence, Garage, Basement...), not real missing data.
- `LotFrontage` is genuinely missing numeric data.
- `Electrical` has a single missing row.
- `SalePrice` is right-skewed and will need a log transform before modeling.

Goal: produce a clean dataset saved to `data/processed/`.

In [2]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 100)

In [3]:
df = pd.read_csv('../data/raw/train.csv')
df.shape

(1460, 81)

## 1. Columns where NA means "feature absent" (categorical -> fill with 'None')

In [4]:
none_cols = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'MasVnrType'
]

for col in none_cols:
    df[col] = df[col].fillna('None')

df[none_cols].isnull().sum().sum()

np.int64(0)

## 2. Columns where NA means "feature absent" (numeric -> fill with 0)

In [5]:
zero_cols = ['GarageYrBlt', 'MasVnrArea']

for col in zero_cols:
    df[col] = df[col].fillna(0)

df[zero_cols].isnull().sum().sum()

np.int64(0)

## 3. LotFrontage: real missing data

Houses in the same neighborhood tend to have similar lot frontage, so we fill by the neighborhood's median instead of a single global median.

In [6]:
df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median())
)
df['LotFrontage'].isnull().sum()

np.int64(0)

## 4. Electrical: single missing row -> fill with mode

In [7]:
df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0])
df['Electrical'].isnull().sum()

np.int64(0)

## 5. Final check: no missing values left

In [8]:
remaining = df.isnull().sum()
remaining[remaining > 0]

Series([], dtype: int64)

## 6. Save the cleaned dataset

In [9]:
df.to_csv('../data/processed/train_clean.csv', index=False)
df.shape

(1460, 81)